# 00 — Setup e teste de conexão

Rode este notebook **antes** de começar. Ele confirma que sua máquina (ou o Colab)
consegue falar com o seu banco Neo4j e que você tem todas as credenciais que o
material vai pedir. Se algo aqui falhar, é bem mais fácil resolver agora do que no
meio do notebook 02 ou 04.

## Criando o banco no AuraDB Free

1. Acesse [console.neo4j.io](https://console.neo4j.io), crie uma conta gratuita
   (sem cartão de crédito) e uma instância **Free**
2. Anote a **URI de conexão**, o **usuário** (geralmente `neo4j`) e a **senha** —
   a senha é mostrada **uma única vez** na criação, salve antes de fechar
3. Anote o **instance id**: é o prefixo da URI. Em
   `neo4j+s://dbd12345.databases.neo4j.io`, o instance id é `dbd12345`
4. Anote o **nome do banco**. ⚠️ Em instâncias AuraDB recentes ele **não** é
   `neo4j`, e sim o próprio instance id. Confira no Console, na página da sua
   instância, ou rode `SHOW DATABASES`

## Credenciais de API (para o notebook 04)

O notebook 04 roda algoritmos de grafo via [Aura Graph
Analytics](https://neo4j.com/docs/aura/graph-analytics/) — sem cobrança no tier
Free, mas com credenciais **diferentes** da senha do banco:

1. No [Aura Console](https://console.neo4j.io), clique no seu perfil/organização
   (canto superior direito) e vá em **API credentials** (ou **API Keys**)
2. Clique em **Create API credentials** e dê um nome qualquer (ex.: `workshop`)
3. Copie o **Client ID** e o **Client Secret** — o secret aparece **uma única vez**

A última célula deste notebook testa essas credenciais, para você não descobrir um
problema só no meio do notebook 04.

## Instalando o driver

Instalamos o **`neo4j-rust-ext`** em vez do `neo4j` puro. Ele não substitui o
driver: **traz o `neo4j` como dependência** e troca a serialização Bolt por uma
implementação em Rust. A API é exatamente a mesma — você continua escrevendo
`from neo4j import GraphDatabase`.

Não desinstale o `neo4j`: o acelerador é um arquivo compilado que vive *dentro*
do pacote `neo4j`.

In [ ]:
!pip install -q neo4j-rust-ext python-dotenv

In [ ]:
from neo4j._codec.packstream import RUST_AVAILABLE

# Se der False, o pip caiu no driver puro Python (normalmente por não haver wheel
# para a sua versão de Python). Tudo funciona igual, só um pouco mais devagar.
print("Extensão Rust ativa:", RUST_AVAILABLE)

## Credenciais

Este notebook busca as credenciais em três lugares, na ordem:

1. **Variáveis de ambiente** — incluindo um arquivo `.env` na pasta do projeto
   (copie o `.env.example` e preencha). É o jeito recomendado: você configura uma
   vez e todos os notebooks funcionam sem digitar nada.
2. **Secrets do Colab** — o ícone de chave 🔑 na barra lateral esquerda. Crie um
   secret por variável (`NEO4J_URI`, `NEO4J_PASSWORD`, …) e ative o acesso para
   este notebook.
3. **Pergunta na tela** — se não achou nas opções acima, pergunta aqui mesmo.

Além de poupar digitação, as duas primeiras opções evitam que a URI da sua
instância fique gravada na saída da célula caso você comite o notebook.

In [ ]:
import os
from getpass import getpass

try:  # carrega um arquivo .env, se existir
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass  # python-dotenv não instalado, ou não há .env — segue o baile


def credencial(nome, prompt, secreta=False, padrao=None):
    """Busca em: variável de ambiente > Secrets do Colab > pergunta na tela."""
    if valor := os.environ.get(nome):
        return valor
    try:
        from google.colab import userdata
        if valor := userdata.get(nome):
            return valor
    except Exception:
        pass  # não é Colab, ou o secret não existe/não foi liberado
    return (getpass(prompt) if secreta else input(prompt)).strip() or padrao


NEO4J_URI = credencial("NEO4J_URI", "URI do Neo4j (ex.: neo4j+s://xxxx.databases.neo4j.io): ")
NEO4J_USER = credencial("NEO4J_USERNAME", "Usuário [neo4j]: ", padrao="neo4j")
NEO4J_PASSWORD = credencial("NEO4J_PASSWORD", "Senha: ", secreta=True)
# Atenção: em instâncias AuraDB recentes o banco NÃO se chama "neo4j", e sim o
# próprio instance id (o prefixo da URI). Confira em Aura Console > sua instância,
# ou rode SHOW DATABASES.
NEO4J_DATABASE = credencial("NEO4J_DATABASE", "Nome do banco (geralmente = instance id) [neo4j]: ", padrao="neo4j")

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Conectado!")

Usamos `driver.execute_query(...)` para rodar cada consulta — a forma mais simples
e direta de mandar uma query pro banco e já receber os resultados de volta, sem
abrir sessão ou transação manualmente. É esse padrão em todos os notebooks.

In [ ]:
records, _, _ = driver.execute_query(
    "CALL dbms.components() YIELD name, versions, edition RETURN name, versions, edition",
    database_=NEO4J_DATABASE,
)
for r in records:
    print(dict(r))

records, _, _ = driver.execute_query("MATCH (n) RETURN count(n) AS total", database_=NEO4J_DATABASE)
print(f"\nO banco tem {records[0]['total']} nós agora "
      f"(deve ser 0 se você acabou de criar a instância).")

### APOC

Os notebooks 02 e 04 usam algumas funções da biblioteca APOC para conferir a carga.
Ela já vem instalada no AuraDB.

In [ ]:
try:
    driver.execute_query("RETURN apoc.version() AS v", database_=NEO4J_DATABASE)
    print("APOC disponível ✅")
except Exception as e:
    print("APOC NÃO disponível — as queries de conferência vão precisar de ajuste. Detalhe:", e)

## Testando as credenciais de API da Aura

Esta é a checagem que evita a pior surpresa do material: descobrir, já no notebook
04, que as credenciais de API estão erradas — depois de esperar dois minutos por
uma sessão que não vai subir.

Pedimos um token de acesso à API da Aura. Se voltar um token, está tudo certo.

Se você ainda não criou as credenciais, pode pular esta célula agora — só não
deixe de voltar aqui antes do notebook 04.

In [ ]:
import json
import urllib.request
import urllib.parse
import base64

AURA_CLIENT_ID = credencial("AURA_CLIENT_ID", "Aura API Client ID (Enter para pular): ", secreta=True)
AURA_CLIENT_SECRET = credencial("AURA_CLIENT_SECRET", "Aura API Client Secret (Enter para pular): ", secreta=True)

if not AURA_CLIENT_ID or not AURA_CLIENT_SECRET:
    print("Pulado. Crie as credenciais no Aura Console antes de rodar o notebook 04.")
else:
    credenciais = base64.b64encode(f"{AURA_CLIENT_ID}:{AURA_CLIENT_SECRET}".encode()).decode()
    requisicao = urllib.request.Request(
        "https://api.neo4j.io/oauth/token",
        data=urllib.parse.urlencode({"grant_type": "client_credentials"}).encode(),
        headers={
            "Authorization": f"Basic {credenciais}",
            "Content-Type": "application/x-www-form-urlencoded",
        },
    )
    try:
        with urllib.request.urlopen(requisicao, timeout=30) as resposta:
            token = json.load(resposta).get("access_token", "")
        if token:
            print(f"Credenciais de API válidas ✅ (token com {len(token)} caracteres)")
            print("O notebook 04 vai conseguir criar a sessão de análise.")
        else:
            print("A API respondeu, mas sem token. Gere novas credenciais no Console.")
    except urllib.error.HTTPError as e:
        print(f"❌ Falha na autenticação (HTTP {e.code}). Client ID/Secret incorretos ou revogados.")
        print("   Gere um novo par em Aura Console > perfil/organização > API credentials.")
    except Exception as e:
        print(f"❌ Não foi possível falar com a API da Aura: {type(e).__name__}: {e}")

## Tudo certo?

Se as células acima rodaram sem erro, o ambiente está pronto. Deixe à mão:

- URI, usuário, senha e **nome do banco**
- **instance id** e as **credenciais de API** (Client ID + Secret)

Cada notebook pede essas informações de novo, porque no Colab cada um roda em seu
próprio kernel. Para não digitar tudo a cada vez, preencha o arquivo `.env` (copie
o `.env.example`) ou use os **Secrets** do Colab — os notebooks procuram lá antes de
perguntar.

In [ ]:
driver.close()